# Segment Analysis: Combined 2017-2019 with Distance Normalization

This notebook:
- Loads ENOA hyperbolic embeddings
- Computes Lorentz distances for collaboration pairs across all years (2017-2019)
- Normalizes distances (min-max and z-score)
- Outputs combined dataset with both raw and normalized distances

In [5]:
import os
os.chdir('..') 
print("Now working from:", os.getcwd())

Now working from: /Users/mikolajandrzejewski/Documents/GitHub


In [6]:
import sys
sys.path.append('embeddings')

import torch
import pickle
import numpy as np
import pandas as pd
from lorentz import Lorentz
from sklearn.preprocessing import MinMaxScaler, StandardScaler

print("Libraries loaded successfully")

Libraries loaded successfully


## Load ENOA embeddings and vocabulary

In [7]:
# Load ENOA checkpoint and vocabulary
vocab = pickle.load(open("enoa_vocab.pkl", "rb"))
genre_to_idx = {genre: idx for idx, genre in enumerate(vocab)}
checkpoint = torch.load("ckpt/final_enoa_graph.ckpt", map_location="cpu")
embeddings = checkpoint["table.weight"].numpy()

print(f"ENOA vocabulary size: {len(vocab)} genres")
print(f"Embedding shape: {embeddings.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'enoa_vocab.pkl'

## Define Lorentz distance function

In [ ]:
def lorentz_scalar_product(u, v):
    """Compute Lorentz scalar product."""
    return -u[0]*v[0] + np.dot(u[1:], v[1:])

def lorentz_distance(u, v):
    """Compute hyperbolic distance in Lorentz model."""
    inner = lorentz_scalar_product(u, v)
    inner = min(inner, -1.0)  # Clipping for numerical stability
    return float(np.arccosh(-inner))

## Process all years (2017-2019)

In [ ]:
years = [2017, 2018, 2019]
all_dfs = []
all_matched_genres = set()
all_unmatched_genres = set()

for year in years:
    print(f"\n{'='*60}")
    print(f"Processing year: {year}")
    print(f"{'='*60}")
    
    # Load collaboration dataset for this year
    df = pd.read_csv(f"code/dataset/Original/global/global-genre_network-{year}.csv", sep="\t")
    
    # Remove self-loops
    df = df[df["source"] != df["target"]].copy()
    collab_genres = set(df["source"].unique()) | set(df["target"].unique())
    print(f"Unique genres in collaboration dataset: {len(collab_genres)}")
    
    # Match genres with ENOA embeddings
    matched = {}
    unmatched = []
    for genre in collab_genres:
        if genre in genre_to_idx:
            idx = genre_to_idx[genre] + 1  # +1 for padding index
            matched[genre] = embeddings[idx]
            all_matched_genres.add(genre)
        else:
            unmatched.append(genre)
            all_unmatched_genres.add(genre)
    
    print(f"Matched: {len(matched)} genres")
    print(f"Unmatched: {len(unmatched)} genres")
    if unmatched:
        print("Unmatched genres:")
        for g in sorted(unmatched):
            print(f"  - {g}")
    
    # Compute distances for each collaboration pair
    distances = []
    skipped_rows = []
    for _, row in df.iterrows():
        source = row["source"]
        target = row["target"]
        
        if source not in matched:
            skipped_rows.append((source, target, f"'{source}' not in ENOA"))
            distances.append(None)
            continue
        if target not in matched:
            skipped_rows.append((source, target, f"'{target}' not in ENOA"))
            distances.append(None)
            continue
        
        d = lorentz_distance(matched[source], matched[target])
        distances.append(d)
    
    df["distance"] = distances
    df["year"] = year
    
    # Remove rows without distances
    df_clean = df.dropna(subset=["distance"]).copy()
    
    print(f"\nTotal collaboration pairs: {len(df)}")
    print(f"Pairs skipped: {len(skipped_rows)}")
    print(f"Pairs with distance computed: {len(df_clean)}")
    
    all_dfs.append(df_clean)

# Combine all years
df_combined = pd.concat(all_dfs, ignore_index=True)

print(f"\n{'='*60}")
print("COMBINED DATASET SUMMARY")
print(f"{'='*60}")
print(f"Total rows across all years: {len(df_combined)}")
print(f"Total unique genres with embeddings: {len(all_matched_genres)}")
print(f"Total unique genres without embeddings: {len(all_unmatched_genres)}")
print(f"\nUnmatched genres across all years:")
for g in sorted(all_unmatched_genres):
    print(f"  - {g}")

## Normalize distances

In [ ]:

distance_stats = df_combined['distance'].describe()
print(f"\nRange: [{distance_stats['min']:.4f}, {distance_stats['max']:.4f}]")

# Min-Max normalization (0-1 range)
min_max_scaler = MinMaxScaler()
df_combined['distance_norm'] = min_max_scaler.fit_transform(df_combined[['distance']])

print("\n" + "="*60)
print("Normalized distances added:")
print("="*60)
print(f"distance_norm: min={df_combined['distance_norm'].min():.4f}, max={df_combined['distance_norm'].max():.4f}")

# Show comparison
print("\nSample comparison (first 10 rows):")
print(df_combined[['source', 'target', 'year', 'distance', 'distance_norm']].head(10))

## Save combined dataset

In [ ]:
# Reorder columns for clarity
column_order = ['year', 'source', 'target', 'weight', 'avg_streams', 
                'distance', 'distance_norm']
df_combined = df_combined[column_order]

# Save to CSV
output_file = "collaboration_with_distances.csv"
df_combined.to_csv(output_file, index=False)

print(f"Combined dataset saved to: {output_file}")
print(f"Shape: {df_combined.shape}")
print(f"\nColumns: {list(df_combined.columns)}")
print(f"\nFirst few rows:")
print(df_combined.head())

## Summary statistics by year

In [ ]:
# Group by year and compute statistics
summary_stats = df_combined.groupby('year').agg({
    'distance': ['count', 'mean', 'std', 'min', 'max'],
    'distance_norm': ['mean', 'std'],
    'weight': ['mean', 'sum'],
    'avg_streams': ['mean', 'median']
}).round(4)

print("Summary statistics by year:")
print(summary_stats)

## Save matched genre embeddings

In [ ]:
# Create final matched embeddings dictionary with all unique genres
final_matched = {}
for genre in all_matched_genres:
    idx = genre_to_idx[genre] + 1
    final_matched[genre] = embeddings[idx]

# Save
pickle.dump(final_matched, open("collab_genre_embeddings.pkl", "wb"))
print(f"Saved {len(final_matched)} genre embeddings to 'collab_genre_embeddings.pkl'")